The best is the Mix model. predict 73% generally
- ill try without the data preprocessing

## 1. Import Library and Dataset

### 1.1 Library

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
import re

from nltk.corpus import stopwords
from tensorflow import keras
from collections import Counter
from spellchecker import SpellChecker

### 1.2 Dataset

dataset: https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment?utm_source=chatgpt.com&library=pandas

In [2]:
splits = {'train': 'sent_train.csv', 'validation': 'sent_valid.csv'}
df = pd.read_csv("hf://datasets/zeroshot/twitter-financial-news-sentiment/" + splits["train"])

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df.head(5)

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


In [4]:
df.shape

(9543, 2)

In [5]:
df['label'].value_counts()

label
2    6178
1    1923
0    1442
Name: count, dtype: int64

0 = bearish; 1 = bullish; 2 = neutral 

## 2. Data Preprocessing

### 2.1 Cleaning Tweets

In [6]:
# Extracting SYmbol
df['symbol'] = df['text'].str.extract(r'(\$[A-Za-z]+)')

In [7]:
# Extract the Text only (remove symbol, mention, and link)
df['text'] = (
    df['text']
    .str.replace(r'\$[A-Za-z]+', ' ', regex=True)          # stock tickers
    .str.replace(r'@[A-Za-z0-9_]+', ' ', regex=True)        # mentions
    .str.replace(r'http\S+|www\.\S+', ' ', regex=True)      # URLs
    .str.replace(r'[^\w\s]', ' ', regex=True)               # remove punctuation safely
    .str.replace(r'\b\w*\d\w*\b', ' ', regex=True)          # remove words w/ numbers
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)                   # collapse extra spaces
    .str.strip()
)

In [ ]:
df = df[['text', 'label']]

# drop row with null value
df.dropna(inplace=True)

# remapping the label
mapping = {
    0: 0,
    1: 2,
    2: 1
}
df['label'] = df['label'].map(mapping)

## 3. Model Selection Algorithm

- pa | ProsusAI/FinBERT: https://huggingface.co/ProsusAI/finbert
- yy | yiyanghkust/finbert-tone: https://huggingface.co/yiyanghkust/finbert-tone
- sa | StephanAkkerman/FinTwitBERT-sentiment: https://huggingface.co/StephanAkkerman/FinTwitBERT
- dr | distilroberta : https://huggingface.co/mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis

### 3.1 Individual Evaluation

In [10]:
from transformers import pipeline
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone',num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')

# Models
models = {
    'pa': pipeline(task="text-classification", model="ProsusAI/finbert"),
    'yy': pipeline("sentiment-analysis", model=finbert, tokenizer=tokenizer),
    'sa': pipeline("sentiment-analysis", model="StephanAkkerman/FinTwitBERT-sentiment"),
    'dr': pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")
    }

for name, pipe in models.items():
    df_sample = df.copy().sample(4000, random_state=42)

    # Run prediction (vectorized)
    preds = pipe(df_sample['text'].tolist())

    # Extract labels
    df_sample['pred_sent_str'] = [p['label'].lower() for p in preds]

    # Map string → int
    sent_map = {"negative": 0, "bearish": 0, "neutral": 1, "positive": 2, "bullish": 2}
    df_sample['pred_sentiment'] = df_sample['pred_sent_str'].map(sent_map)

    # Compare with true labels
    df_sample['correct'] = (df_sample['label'] == df_sample['pred_sentiment']).astype(int)

    # Accuracy
    accuracy = df_sample['correct'].mean()
    print(f"Accuracy - {name}: {accuracy:.4f}")

Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu


Accuracy - pa: 0.6900
Accuracy - yy: 0.7202
Accuracy - sa: 0.8035
Accuracy - dr: 0.7302


### 3.2 Mix Model

In [11]:
def custom_majority(row):
    vals = row.tolist()
    
    # Case: all different → choose 1 (neutral)
    if len(set(vals)) == 3:
        return 1
    
    # Otherwise normal majority vote
    return row.mode()[0]

# Models
models = {
    'yy': pipeline("sentiment-analysis", model=finbert, tokenizer=tokenizer),
    'sa': pipeline("sentiment-analysis", model="StephanAkkerman/FinTwitBERT-sentiment"),
    'dr': pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")
}

# Sample ONCE so all models use the same rows
df_eval = df.copy().sample(4000, random_state=42)[['text', 'label']]

# Mapping
sent_map = {"negative": 0, "bearish": 0, "neutral": 1, "positive": 2, "bullish": 2}

# Run each model on the same rows
for name, pipe in models.items():
    preds = pipe(df_eval['text'].tolist())
    df_eval[f'pred_{name}'] = [sent_map[p['label'].lower()] for p in preds]

# Majority voting
cols = [f'pred_{key}' for key in models.keys()]
df_eval['majority'] = df_eval[cols].apply(custom_majority, axis=1)

# Accuracy
df_eval['correct'] = (df_eval['label'] == df_eval['majority']).astype(int)
accuracy = df_eval['correct'].mean()

print(f"Ensemble Accuracy: {accuracy:.4f}")

Device set to use cpu
Device set to use cpu
Device set to use cpu


Ensemble Accuracy: 0.7953
